
#Note:
1. Pydantic V1 → V2 Important API Changes

In [ ]:
# Pydantic V1                 Pydantic V2
# .dict()                     .model_dump()
# .json()                     .model_dump_json()
# .parse_obj()                .model_validate()
# .parse_raw()                .model_validate_json()
# .schema()                   .model_json_schema()
# .copy()                     .model_copy()
# @validator                  @field_validator
# @root_validator             @model_validator

2. Basic BaseModel

In [6]:
from pydantic import BaseModel

In [7]:
class User(BaseModel):
    name: str
    age: int

Users= User(name="DJ", age=40)

print(Users)
print(Users.name)
print(Users.age)


name='DJ' age=40
DJ
40


#3. Required Field

In [8]:
class User(BaseModel):
    name: str
    
users = User(name="DJ")
print(users)

name='DJ'


#4. Field With Default Value

In [9]:
class User(BaseModel):
    name: str
    country: str = "India"

user = User(name="DJ")
print(user)

name='DJ' country='India'


#5. Optional-To-Provide + Nullable Field

In [10]:
class User(BaseModel):
    middle_name: str|None = None

users = User()
print(users)

users = User(middle_name="DJ")
print(users)

middle_name=None
middle_name='DJ'


#6. Type Coercion / Parsing

In [11]:
class Employee(BaseModel):
    employee_id: int

employee = Employee(employee_id="123")
print(employee)
print(type(employee.employee_id))

employee_id=123
<class 'int'>


#7. Invalid Type ()

In [12]:
class Employee(BaseModel):
    employee_id: int

employee = Employee(employee_id="DJ")


ValidationError: 1 validation error for Employee
employee_id
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='DJ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

## 8.model_validate()

## If the object could not be validated. then raise ValidationError

In [13]:
class User(BaseModel):
    name: str
    age: int

data = {"name": "DJ", "age": "40"}
user = User.model_validate(data)
print(user)

data = {"name": "DJ", "age": "DJ"}
user = User.model_validate(data)
print(user)

name='DJ' age=40


ValidationError: 1 validation error for User
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='DJ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

### 9. model_validate_json()
### Raises: ValidationError: If json_data is not a JSON string or the object could not be validated.

In [14]:
class User(BaseModel):
    name: str
    age: int

json_data  = '{"name": "DJ", "age": 40}'
user = User.model_validate_json(json_data)
print(user)




name='DJ' age=40


### 10. model_validate_strings()
### Validate the given object with string data against the Pydantic model.

In [15]:
class User(BaseModel):
    name: str
    age: int

data = {"name": "DJ", "age": "40"}
users = User.model_validate_strings(data)
print(users)

name='DJ' age=40


### 11. model_construct() — Skip Validation

In [16]:
class User(BaseModel):
    name: str
    age: int

user = User.model_construct(name="DJ", age="not-an-int")
print(user)

name='DJ' age='not-an-int'


### 12. model_rebuild()

In [17]:
from __future__ import annotations

class Employee(BaseModel):
    name: str
    manager: Employee | None = None

Employee.model_rebuild()

### 13. Basic Field()
Useful constraints:
Field(gt=0)

greater than 0.

Field(ge=0)

greater than or equal to 0.

Field(lt=100)

less than 100.

Field(le=100)

less than or equal to 100.

Field(multiple_of=5)

must be a multiple of 5.

For strings:

Field(min_length=3)

minimum length of 3 characters.

Field(max_length=50)

maximum length of 50 characters.

Field(min_length=3, max_length=50)

length must be between 3 and 50 characters.

Field(pattern=r"^[a-zA-Z]+$")

must match the given regular expression.

For lists/collections:

Field(min_length=1)

minimum length of 1 item.

Field(max_length=10)

maximum length of 10 items.

Field(min_length=1, max_length=10)

length must be between 1 and 10 items.

For decimals:

Field(max_digits=10)

maximum of 10 digits.

Field(decimal_places=2)

maximum of 2 decimal places.

Field(max_digits=10, decimal_places=2)

maximum of 10 digits with at most 2 decimal places.

In [18]:
from pydantic import BaseModel, Field

class Employee(BaseModel):
    name: str
    age: int = Field(ge=18, le=65)

In [19]:
Emp1 = Employee(name="DJ", age=40)
print(Emp1)

#getting ValidationError
Emp1 = Employee(name="DJ", age=15)
print(Emp1)

name='DJ' age=40


ValidationError: 1 validation error for Employee
age
  Input should be greater than or equal to 18 [type=greater_than_equal, input_value=15, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal

## 14. Numeric Constraints

In [24]:
from pydantic import BaseModel, fields

In [27]:
class Product(BaseModel):
    price: float = Field(gt = 0)
    quantity: int = Field(ge = 0)
    discount: float = Field(ge=0, le=100)
    pack_size: int = Field(multiple_of = 5)


## 15. String Constraints

In [28]:
from pydantic import BaseModel, fields

class user(BaseModel):
    username: str = Field(min_length=3, max_length=30)
    code: str = Field(pattern=r"^[A-Z]{3}\d{3}$")

## 16. Decimal Constraints

In [35]:
from decimal import Decimal
from pydantic import BaseModel,fields

class Product(BaseModel):
    price: Decimal = Field(max_digits=8, decimal_places=2)



In [38]:
product = Product(price="12345.67")
print(product)


price=Decimal('12345.67')


## 17. allow_inf_nan

In [45]:
from pydantic import BaseModel, Field

class Metrics(BaseModel):
    score: float = Field(allow_inf_nan=False)

In [46]:
Metrics(score="nan")

ValidationError: 1 validation error for Metrics
score
  Input should be a finite number [type=finite_number, input_value='nan', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/finite_number

## 18. Annotated

In [47]:
from typing import Annotated
from pydantic import BaseModel, Field

Age = Annotated[int, Field(ge=18, le=100)]

class User(BaseModel):
    name: str
    age: Age

In [48]:
User(name="dj",age=18)

User(name='dj', age=18)

In [49]:
User(name="dj",age=17)

ValidationError: 1 validation error for User
age
  Input should be greater than or equal to 18 [type=greater_than_equal, input_value=17, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal

## 19. Reusable Annotated Types

In [50]:

from typing import Annotated
from pydantic import BaseModel, Field

PositivePrice = Annotated[float, Field(gt=0)]

class Product(BaseModel):
    name: str
    price: PositivePrice

class Service(BaseModel):
    name: str
    price: PositivePrice

## 20. Simple Default Value

In [51]:
from pydantic import BaseModel

class User(BaseModel):
    status: str = "active"

user = User()
print(user)

status='active'


## 21. default_factory

In [54]:
from datetime import datetime
from pydantic import BaseModel, Field

class Event(BaseModel):
    created_at: datetime = Field(default_factory=datetime.now)
    tags: list[str] = Field(default_factory=list)

event = Event()
print(event)

created_at=datetime.datetime(2026, 9, 22, 14, 40, 11, 266350) tags=[]
